**Scraper for Canadian National Parks**

In [ ]:
import json
import time
import datetime
import requests
from bs4 import BeautifulSoup

# Comprehensive target URL list for Canadian National Parks
parks_canada_urls = [
    # 1 Banff National Park
    "https://parks.canada.ca/pn-np/ab/banff",
    "https://parks.canada.ca/pn-np/ab/banff/visit/les10-top10",
    "https://parks.canada.ca/pn-np/ab/banff/visit/parkbus/calgary",
    "https://parks.canada.ca/pn-np/ab/banff/visit/les10-top10/promenade-de-la-vallee-de-la-bow-bow-valley-parkway",

    # 2 Pacific Rim National Park Reserve
    "https://parks.canada.ca/pn-np/bc/pacificrim",
    "https://parks.canada.ca/pn-np/bc/pacificrim/activ/sco-wct",
    "https://parks.canada.ca/pn-np/bc/pacificrim/activ",
    "https://parks.canada.ca/pn-np/bc/pacificrim/activ/surf",
    "https://parks.canada.ca/pn-np/bc/pacificrim/activ/broken-group",

    # 3 Saguenay–St. Lawrence Marine Park
    "https://parks.canada.ca/amnc-nmca/qc/saguenay",
    "https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/decouverte-tours",
    "https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/observation-watching",
    "https://parks.canada.ca/amnc-nmca/qc/saguenay/visit/cdmm",

    # 4 Jasper National Park
    "https://parkscanada.ca/jasper",
    "https://parks.canada.ca/pn-np/ab/jasper/activ/experience/printemps-spring",
    "https://parks.canada.ca/pn-np/ab/jasper/nature/conservation/retablissement-caribou-recovery",
    "https://parks.canada.ca/pn-np/ab/jasper/sources-miette-springs",

    # 5 Mount Revelstoke & Glacier NPs
    "https://parks.canada.ca/pn-np/bc/glacier",
    "https://parks.canada.ca/pn-np/bc/glacier/activ/randonee-hiking",
    "https://parks.canada.ca/pn-np/bc/glacier/visit/centre",

    # 6 Yoho National Park
    "https://parks.canada.ca/pn-np/bc/yoho",
    "https://parks.canada.ca/pn-np/bc/yoho/activ/guidee-conservation-guided",
    "https://parks.canada.ca/pn-np/bc/yoho/activ",
    "https://parks.canada.ca/pn-np/bc/yoho/activ/randonnee-hike",
    "https://parks.canada.ca/pn-np/bc/yoho/activ/randonnee-hike/ohara/visit",
    "https://parks.canada.ca/pn-np/bc/yoho/activ/places",

    # 7 Kootenay National Park
    "https://parks.canada.ca/pn-np/bc/kootenay",
    "https://parks.canada.ca/pn-np/bc/kootenay/sources-radium-springs",
    "https://parks.canada.ca/pn-np/bc/kootenay/activ/randonnee-hike/etat-sentiers-trail-conditions",
    "https://parks.canada.ca/pn-np/bc/kootenay/securite-safety",

    # 8 Waterton Lakes National Park
    "https://parks.canada.ca/pn-np/ab/waterton",
    "https://parks.canada.ca/pn-np/ab/waterton/activ",
    "https://parks.canada.ca/pn-np/ab/waterton/activ/experiences",
    "https://parks.canada.ca/pn-np/ab/waterton/activ/experiences/randonee-hiking",
    "https://parks.canada.ca/pn-np/ab/waterton/activ/experiences/tranquille-quiet",
    "https://parks.canada.ca/pn-np/ab/waterton/activ/experiences/observationfaune-wildlifeviewing",

    # 9 Point Pelee National Park
    "https://parks.canada.ca/pn-np/on/pelee",
    "https://parks.canada.ca/pn-np/on/pelee/activ",
    "https://parks.canada.ca/pn-np/on/pelee/activ/pagayez-paddling",
    "https://parks.canada.ca/pn-np/on/pelee/activ/randonnee-hiking",
    "https://parks.canada.ca/pn-np/on/pelee/activ/oiseaux-birds",
    "https://parks.canada.ca/pn-np/on/pelee/activ/velo-bicycling",

    # 10 Bruce Peninsula National Park
    "https://parks.canada.ca/pn-np/on/bruce",
    "https://parks.canada.ca/pn-np/on/bruce/activ",
    "https://parks.canada.ca/pn-np/on/bruce/activ/emplacements-locations",
    "https://parks.canada.ca/pn-np/on/bruce/activ/sentiers-trails"
]

def parse_html_table_to_markdown(table_tag):
    """Transforms a standard HTML table element into a relational Markdown text structure."""
    markdown_lines = []
    rows = table_tag.find_all("tr")
    if not rows:
        return ""

    # Process headers
    headers = [th.get_text().strip() for th in rows[0].find_all(["th", "td"])]
    if headers:
        markdown_lines.append("| " + " | ".join(headers) + " |")
        markdown_lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    # Process body data rows
    start_idx = 1 if len(rows) > 1 else 0
    for row in rows[start_idx:]:
        cells = [td.get_text().strip() for td in row.find_all("td")]
        if not cells:
            # Fallback check if headers were nested deep within rows
            cells = [th.get_text().strip() for th in row.find_all("th")]
        if cells:
            markdown_lines.append("| " + " | ".join(cells) + " |")

    return "\n".join(markdown_lines)

def extract_clean_text(soup):
    # Strip out non-content structural tags
    for element in soup(["script", "style", "nav", "footer", "header", "form", "input", "button"]):
        element.decompose()

    # Target the Parks Canada layout-specific content main wrapper
    main_content = soup.find("main")

    if not main_content:
        main_content = soup.body if soup.body else soup

    # Form text block list by checking elements recursively or directly
    text_blocks = []

    # Text lines we explicitly want to exclude from contaminating our data blocks
    blacklisted_phrases = {
        "go to province / territory", "newfoundland and labrador", "nova scotia",
        "prince edward island", "new brunswick", "quebec", "ontario", "manitoba",
        "saskatchewan", "alberta", "british columbia", "northwest territories", "yukon",
        "buy your discovery pass online", "buy your pass in person", "contact us"
    }

    # Find text tags and table elements chronologically in the DOM flow
    for element in main_content.find_all(["h1", "h2", "h3", "p", "li", "table"]):
        # 1. Check if the element is an explicit data table
        if element.name == "table":
            md_table = parse_html_table_to_markdown(element)
            if md_table:
                text_blocks.append(md_table)
            continue

        # 2. Otherwise process it as standard semantic text
        if element.find_parent("table"):
            continue

        cleaned_text = element.get_text().strip()
        if not cleaned_text:
            continue

        # Skip noisy dropdown strings or navigation placeholders
        if cleaned_text.lower() in blacklisted_phrases or len(cleaned_text) < 2:
            continue

        text_blocks.append(cleaned_text)

    # Reassemble using double newlines for optimal text-splitting context limits
    return "\n\n".join(text_blocks)

def run_scraper():
    headers = {
        "User-Agent": "EducationalRAGBot/1.0 (Contact: student_learning_rag@example.com)"
    }
    scraped_dataset = []

    print("Starting collection for Parks Canada...")
    for url in parks_canada_urls:
        print(f"Scraping: {url}")
        try:
            # We use allow_redirects=True to safely handle any shortlinks like parkscanada.ca/jasper
            response = requests.get(url, headers=headers, timeout=15, allow_redirects=True)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")
                title = soup.title.string.strip() if soup.title else "Untitled Page"
                clean_text = extract_clean_text(soup)

                scraped_dataset.append(
                    {
                        "title": title,
                        "url": response.url, # Saves the final URL in case of redirect
                        "source": "Canada",
                        "content": clean_text,
                        "scraped_at": datetime.datetime.now(datetime.UTC).isoformat(),
                    }
                )
            else:
                print(f"Failed to fetch {url}. Status code: {response.status_code}")
        except Exception as e:
            print(f"An error occurred while scraping {url}: {str(e)}")

        # 2-second polite throttle
        time.sleep(2)

    # Updated Output Filename
    output_filename = "canada-top-parks.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(scraped_dataset, f, ensure_ascii=False, indent=4)

    print(f"\nSuccessfully saved {len(scraped_dataset)} pages to '{output_filename}'!")

# Run the updated execution setup
run_scraper()

Starting collection for Parks Canada...
Scraping: https://parks.canada.ca/pn-np/ab/banff
Scraping: https://parks.canada.ca/pn-np/ab/banff/visit/les10-top10
Scraping: https://parks.canada.ca/pn-np/ab/banff/visit/parkbus/calgary
Scraping: https://parks.canada.ca/pn-np/ab/banff/visit/les10-top10/promenade-de-la-vallee-de-la-bow-bow-valley-parkway
Scraping: https://parks.canada.ca/pn-np/bc/pacificrim
Scraping: https://parks.canada.ca/pn-np/bc/pacificrim/activ/sco-wct
Scraping: https://parks.canada.ca/pn-np/bc/pacificrim/activ
Scraping: https://parks.canada.ca/pn-np/bc/pacificrim/activ/surf
Scraping: https://parks.canada.ca/pn-np/bc/pacificrim/activ/broken-group
Scraping: https://parks.canada.ca/amnc-nmca/qc/saguenay
Scraping: https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/decouverte-tours
Scraping: https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/observation-watching
Scraping: https://parks.canada.ca/amnc-nmca/qc/saguenay/visit/cdmm
Scraping: https://parkscanada.ca/jasper
Scraping:

Checking the data

In [ ]:
with open("canada-top-parks.json", "r", encoding="utf-8") as f:
    preview_data = json.load(f)

# Let's inspect Document 2 (index 1) to verify our markdown conversion strategy
print(f"Page Title: {preview_data[1]['title']}")
print("\nFirst 600 characters of clean content:")
print("-" * 40)
print(preview_data[1]['content'][:600])
print("-" * 40)

Page Title: What to see while visiting Banff National Park - Banff National Park

First 600 characters of clean content:
----------------------------------------
What to see while visiting Banff National Park

Banff National Park

There are so many things to see in Banff National Park that it's sometimes difficult to know where to start! Here are some of the highlights. Keep in mind, the park is especially busy from May through September.

There are so many things to see in Banff National Park that it's sometimes difficult to know where to start! Here are some of the highlights. Keep in mind, the park is especially busy from May through September.

Use BanffNow for real-time congestion information and parking availability. Better yet, take transit or a
----------------------------------------


In [ ]:
with open("canada-top-parks.json", "r") as f:
    preview_data = json.load(f)
    print(f"Total documents collected: {len(preview_data)}")
    print(f"First Page Title: {preview_data[0]['title']}")
    print(f"Content Snippet:\n{preview_data[0]['content'][:300]}...")

Total documents collected: 46
First Page Title: Banff National Park
Content Snippet:
Banff National Park

Rocky Mountain peaks, glacial lakes, and adventure come together in Banff National Park - Canada’s first national park and the flagship of the nation’s park system.  Banff is part of the Canadian Rocky Mountain Parks UNESCO World Heritage Site.

Rocky Mountain peaks, glacial lak...


Cleaning Canadian parks data

In [ ]:
import json

def clean_scraped_dataset(input_filename, output_filename):
    print(f"Loading raw data from '{input_filename}'...")

    with open(input_filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    # The expanded blacklist of UI/Navigation noise
    blacklisted_phrases = {
        "go to province / territory", "newfoundland and labrador", "nova scotia",
        "prince edward island", "new brunswick", "quebec", "ontario", "manitoba",
        "saskatchewan", "alberta", "british columbia", "northwest territories", "yukon",
        "buy your discovery pass online", "buy your pass in person", "contact us",
        "facebook", "twitter", "instagram", "youtube", "more on hours of operation",
        "more contact information", "@banffnp / #banffnp", "current widlfire status",
        "bow valley parkway", "bear safety", "water activities", "how to get here",
        "banff national park management", "#wildliferules", "park regulations",
        "important bulletins", "most requested", "related links", "hours and fees"
    }

    cleaned_data = []

    for item in data:
        raw_content = item["content"]

        # Split the text content into individual lines
        raw_lines = raw_content.split('\n')

        seen_lines = set()
        unique_lines = []

        for line in raw_lines:
            cleaned_line = line.strip()
            cleaned_lower = cleaned_line.lower()

            # 1. Skip completely empty lines
            if not cleaned_line:
                continue

            # 2. Skip explicitly blacklisted UI phrases
            if cleaned_lower in blacklisted_phrases:
                continue

            # 3. Filter out fragmented noise (short strings)
            # Exception: Keep short strings if they are part of a Markdown table (|),
            # a phone number (digits), or an email (@)
            if len(cleaned_line) < 15:
                if not ('|' in cleaned_line or any(char.isdigit() or char == '@' for char in cleaned_line)):
                    continue

            # 4. Strict Deduplication (preserving the exact order of the page)
            if cleaned_lower not in seen_lines:
                seen_lines.add(cleaned_lower)
                unique_lines.append(cleaned_line)

        # Re-assemble the clean text using double-newlines for the LLM
        item["content"] = "\n\n".join(unique_lines)
        cleaned_data.append(item)

    # Save the polished data to a new file
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

    print(f"Cleanup complete! Optimized {len(cleaned_data)} pages and saved to '{output_filename}'.")


# Execute the cleanup
clean_scraped_dataset("canada-top-parks.json", "canada-parks-cleaned.json")

Loading raw data from 'canada-top-parks.json'...
Cleanup complete! Optimized 46 pages and saved to 'canada-parks-cleaned.json'.


**Scraper for U.S. National Parks**

In [ ]:
import json
import time
import datetime
import requests
from bs4 import BeautifulSoup

# U.S. National Parks URL list
nps_urls = [
    # Great Smoky Mountains
    "https://www.nps.gov/grsm/faqs.htm", "https://www.nps.gov/grsm/index.htm",
    "https://www.nps.gov/grsm/planyourvisit/big-creek.htm", "https://www.nps.gov/grsm/planyourvisit/cataloochee-balsam.htm",
    # Zion
    "https://www.nps.gov/zion/index.htm", "https://www.nps.gov/zion/frequently-asked-questions-about-zion-canyon.htm",
    "https://www.nps.gov/zion/frequently-asked-questions-about-zions-hiking-trails.htm",
    "https://www.nps.gov/zion/frequently-asked-questions-about-the-narrows-virgin-river.htm",
    "https://www.nps.gov/zion/frequently-asked-questions-about-kolob-canyons.htm",
    # Grand Canyon
    "https://www.nps.gov/grca/learn/management/statistics.htm",
    "https://www.nps.gov/grca/planyourvisit/things2do.htm", "https://www.nps.gov/grca/planyourvisit/fees.htm",
    # Yellowstone
    "https://www.nps.gov/yell/planyourvisit/placestogo.htm", "https://www.nps.gov/yell/faqs.htm",
    "https://www.nps.gov/yell/learn/nature/volcano.htm", "https://www.nps.gov/yell/learn/nature/hydrothermal-systems.htm",
    # Rocky Mountain
    "https://www.nps.gov/romo/learn/tribal-partners.htm", "https://www.nps.gov/romo/learn/nature/mammals.htm",
    "https://www.nps.gov/romo/plan-your-summer-trip-to-rocky.htm", "https://www.nps.gov/romo/plan-your-fall-trip-to-rocky.htm",
    "https://www.nps.gov/romo/planyourvisit/holzwarth-historic-site.htm",
    # Yosemite
    "https://www.nps.gov/yose/index.htm", "https://www.nps.gov/yose/planyourvisit/yv.htm",
    "https://www.nps.gov/yose/planyourvisit/hh.htm", "https://www.nps.gov/yose/faqs.htm",
    # Acadia
    "https://www.nps.gov/acad/learn/nature/index.htm", "https://www.nps.gov/acad/planyourvisit/park-loop-road.htm",
    "https://www.nps.gov/acad/planyourvisit/west-side-mountains-lakes.htm", "https://www.nps.gov/acad/planyourvisit/hiking.htm",
    # Olympic
    "https://www.nps.gov/olym/planyourvisit/directions.htm", "https://www.nps.gov/olym/planyourvisit/visiting-the-elwha.htm",
    "https://www.nps.gov/olym/planyourvisit/visiting-the-hoh.htm", "https://www.nps.gov/olym/planyourvisit/visiting-hurricane-ridge.htm",
    # Grand Teton
    "https://www.nps.gov/grte/faqs.htm", "https://www.nps.gov/grte/planyourvisit/colterbayplan.htm",
    "https://www.nps.gov/grte/planyourvisit/mooseplan.htm", "https://www.nps.gov/grte/planyourvisit/jennylakeplan.htm",
    # Glacier
    "https://www.nps.gov/glac/faqs.htm", "https://www.nps.gov/glac/planyourvisit/hikinglakemcdonald.htm",
    "https://www.nps.gov/glac/planyourvisit/hikingmanyglacier.htm", "https://www.nps.gov/glac/planyourvisit/guided-winter-activities.htm"
]

def extract_nps_content(soup):
    # Remove UI noise
    for element in soup(["script", "style", "nav", "footer", "header", "form", "input", "button", "aside"]):
        element.decompose()

    # Target NPS specific content containers
    main_content = soup.find("div", {"id": "main"}) or soup.find("div", {"class": "ColumnMain"})
    if not main_content:
        main_content = soup.body

    text_blocks = []
    # Extract semantic text
    for element in main_content.find_all(["h1", "h2", "h3", "p", "li"]):
        text = element.get_text().strip()
        if len(text) > 15: # Filter out short UI fragments
            text_blocks.append(text)

    # Deduplicate while preserving order
    seen = set()
    unique = []
    for t in text_blocks:
        if t.lower() not in seen:
            seen.add(t.lower())
            unique.append(t)

    return "\n\n".join(unique)

def run_nps_scraper():
    scraped_data = []
    headers = {"User-Agent": "NPS-Data-Collector/1.0"}

    for url in nps_urls:
        print(f"Scraping: {url}")
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, "html.parser")
                scraped_data.append({
                    "title": soup.title.string.strip() if soup.title else "Untitled",
                    "url": url,
                    "source": "US",
                    "content": extract_nps_content(soup),
                    "scraped_at": datetime.datetime.now(datetime.UTC).isoformat()
                })
        except Exception as e:
            print(f"Error {url}: {e}")
        time.sleep(2) # Polite delay

    with open("us-top-parks.json", "w", encoding="utf-8") as f:
        json.dump(scraped_data, f, indent=4)
    print("Done! Data saved to us-top-parks.json")

run_nps_scraper()

Scraping: https://www.nps.gov/grsm/faqs.htm
Scraping: https://www.nps.gov/grsm/index.htm
Scraping: https://www.nps.gov/grsm/planyourvisit/big-creek.htm
Scraping: https://www.nps.gov/grsm/planyourvisit/cataloochee-balsam.htm
Scraping: https://www.nps.gov/zion/index.htm
Scraping: https://www.nps.gov/zion/frequently-asked-questions-about-zion-canyon.htm
Scraping: https://www.nps.gov/zion/frequently-asked-questions-about-zions-hiking-trails.htm
Scraping: https://www.nps.gov/zion/frequently-asked-questions-about-the-narrows-virgin-river.htm
Scraping: https://www.nps.gov/zion/frequently-asked-questions-about-kolob-canyons.htm
Scraping: https://www.nps.gov/grca/learn/management/statistics.htm
Scraping: https://www.nps.gov/grca/planyourvisit/things2do.htm
Scraping: https://www.nps.gov/grca/planyourvisit/fees.htm
Scraping: https://www.nps.gov/yell/planyourvisit/placestogo.htm
Scraping: https://www.nps.gov/yell/faqs.htm
Scraping: https://www.nps.gov/yell/learn/nature/volcano.htm
Scraping: https:

Checking the data

In [ ]:
with open("us-top-parks.json", "r") as f:
    preview_data = json.load(f)
    print(f"Total documents collected: {len(preview_data)}")
    print(f"First Page Title: {preview_data[0]['title']}")
    print(f"Content Snippet:\n{preview_data[0]['content'][:300]}...")

Total documents collected: 41
First Page Title: Frequently Asked Questions - Great Smoky Mountains National Park (U.S. National Park Service)
Content Snippet:
Frequently Asked Questions

Where can I get information to plan my trip to the park?

The best way to plan your trip is by starting on the "Plan Your Visit" page. From this page, you can find different places in the park that interest you and create your own itinerary.

Will my cell phone work in th...


In [ ]:
with open("us-top-parks.json", "r", encoding="utf-8") as f:
    preview_data = json.load(f)

# Let's inspect Document 2 (index 1) to verify our markdown conversion strategy
print(f"Page Title: {preview_data[1]['title']}")
print("\nFirst 600 characters of clean content:")
print("-" * 40)
print(preview_data[1]['content'][:600])
print("-" * 40)

Page Title: Great Smoky Mountains National Park (U.S. National Park Service)

First 600 characters of clean content:
----------------------------------------
A Wondrous Diversity of Life

Ridge upon ridge of forest straddles the border between North Carolina and Tennessee in Great Smoky Mountains National Park. World renowned for its diversity of plant and animal life, the beauty of its ancient mountains, and the quality of its remnants of Southern Appalachian mountain culture, this is America's most visited national park. Plan your visit today!

A parking tag is required to park for longer than 15 minutes.

Know the status of the park's roads, facilities, and trails before you go.

Learn about the many areas of the park and plan ahead for an enjoy
----------------------------------------


## Vector Database pipeline

*Installing chromadb*

In [ ]:
!pip install chromadb sentence-transformers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:
# Vector Database pipeline
# Free, lightweight setup: structure-aware chunking + all-MiniLM-L6-v2 + Chroma

import json
import os
import re
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

# =========================
# Configuration
# =========================
INPUT_FILES = [
    "canada-top-parks.json",
    "us-top-parks.json",
]

CHROMA_DIR = "chroma_parks_db"
COLLECTION_NAME = "parks_rag_v1"
RESET_COLLECTION = True
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 900
CHUNK_OVERLAP = 120
MIN_CHUNK_LEN = 80

# sentence-transformers/all-MiniLM-L6-v2 - why this embedding model:
# - Free and open source
# - Lightweight and fast for Streamlit deployment
# - Good baseline semantic-search quality for documentation-style park content
# - 384-dimensional vectors keep storage small and retrieval efficient

# # 1) Contextual header injection: each chunk text is prefixed with the page title.
#    This helps the LLM identify the park/page directly from retrieved context.
# 2) Chunk overlap remains 120 for now. It is a sensible default for this dataset.
#    If facts appear split during retrieval tests, should be increased to 200 later.
# 3) Vector DB integrity: collection versioning + optional reset are included to avoid
#    zombie chunks.


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_preserve_tables(text: str):
    text = str(text).replace("", "")
    parts = []
    table_pattern = re.compile(r'(?:^|)(?:[^]*\|[^]*){2,}', re.M)
    pos = 0

    for match in table_pattern.finditer(text):
        before = text[pos:match.start()]
        table = match.group(0)
        if before.strip():
            parts.append(("text", before.strip()))
        parts.append(("table", table.strip()))
        pos = match.end()

    remainder = text[pos:]
    if remainder.strip():
        parts.append(("text", remainder.strip()))

    if not parts and text.strip():
        parts.append(("text", text.strip()))

    return parts


def recursive_sentence_chunk(text: str, chunk_size: int = 900, overlap: int = 120):
    text = normalize_text(text)
    if len(text) <= chunk_size:
        return [text]

    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= chunk_size:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)
            if len(sentence) <= chunk_size:
                current = sentence
            else:
                step = max(1, chunk_size - overlap)
                for i in range(0, len(sentence), step):
                    piece = sentence[i:i + chunk_size].strip()
                    if piece:
                        chunks.append(piece)
                current = ""

    if current:
        chunks.append(current)

    return chunks


def add_context_header(title: str, chunk_text: str) -> str:
    title = normalize_text(title)
    chunk_text = normalize_text(chunk_text)
    return f"{title}\n\n{chunk_text}"

def load_documents(input_files):
    rows = []
    for file_path in input_files:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for i, item in enumerate(data):
            rows.append({
                "doc_id": f"{Path(file_path).stem}_{i}",
                "file": file_path,
                "title": item.get("title", ""),
                "url": item.get("url", ""),
                "source": item.get("source", ""),
                "content": item.get("content", ""),
            })

    return pd.DataFrame(rows)


def build_chunks(documents_df):
    chunks = []

    for _, row in documents_df.iterrows():
        chunk_num = 0
        parts = split_preserve_tables(row["content"])

        for part_type, part_text in parts:
            if part_type == "table":
                piece_list = [normalize_text(part_text)]
            else:
                piece_list = recursive_sentence_chunk(
                    part_text,
                    chunk_size=CHUNK_SIZE,
                    overlap=CHUNK_OVERLAP,
                )

            for piece in piece_list:
                if len(piece) < MIN_CHUNK_LEN:
                    continue

                contextual_text = add_context_header(row["title"], piece)

                chunks.append({
                    "id": f"{row['doc_id']}_c{chunk_num}",
                    "doc_id": row["doc_id"],
                    "file": row["file"],
                    "title": row["title"],
                    "url": row["url"],
                    "source": row["source"],
                    "text": contextual_text,
                    "raw_text": normalize_text(piece),
                    "chars": len(contextual_text),
                })
                chunk_num += 1

    return pd.DataFrame(chunks)


def save_outputs(chunks_df, output_dir="chunks_embeddings_outputs"):
    os.makedirs(output_dir, exist_ok=True)

    chunks_csv = os.path.join(output_dir, "parks_chunks.csv")
    summary_csv = os.path.join(output_dir, "parks_summary.csv")

    chunks_df.to_csv(chunks_csv, index=False)

    summary_df = pd.DataFrame([
        {
            "documents": chunks_df["doc_id"].nunique(),
            "chunks_created": len(chunks_df),
            "avg_chunk_chars": round(chunks_df["chars"].mean(), 2),
            "embedding_model": EMBED_MODEL,
            "vector_db": "Chroma",
            "collection_name": COLLECTION_NAME,
            "reset_collection": RESET_COLLECTION,
        }
    ])
    summary_df.to_csv(summary_csv, index=False)

    return chunks_csv, summary_csv


def create_embeddings(chunks_df):
    print(f"Loading embedding model: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)

    print("Creating embeddings...")
    embeddings = model.encode(
        chunks_df["text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
    ).tolist()

    return model, embeddings


def get_or_reset_collection(client, collection_name: str, reset: bool = False):
    if reset:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except Exception:
            pass
    return client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )


def store_in_chroma(chunks_df, embeddings):
    print(f"Creating/using Chroma DB at: {CHROMA_DIR}")
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    collection = get_or_reset_collection(client, COLLECTION_NAME, reset=RESET_COLLECTION)

    collection.upsert(
        ids=chunks_df["id"].tolist(),
        documents=chunks_df["text"].tolist(),
        embeddings=embeddings,
        metadatas=chunks_df[["title", "url", "source", "file", "doc_id"]].to_dict(orient="records"),
    )

    print(f"Stored {collection.count()} chunks in Chroma collection: {COLLECTION_NAME}")
    return client, collection


def test_query(model, collection, query="What parks mention camping discounts or park passes?"):
    print(f"\nTest query: {query}")
    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=5,
    )

    for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), start=1):
        print(f"\nResult {i}")
        print("Title:", meta.get("title"))
        print("Source:", meta.get("source"))
        print("File:", meta.get("file"))
        print("URL:", meta.get("url"))
        print("Chunk Preview:", doc[:350], "...")

    return results


In [ ]:
# =========================
# Run pipeline
# =========================
print("Loading source documents...")
documents_df = load_documents(INPUT_FILES)
print(f"Loaded {len(documents_df)} source documents")

print("Building chunks...")
chunks_df = build_chunks(documents_df)
print(f"Created {len(chunks_df)} chunks")
print(chunks_df[["id", "title", "chars"]].head())

chunks_csv, summary_csv = save_outputs(chunks_df)
print(f"Saved chunks CSV: {chunks_csv}")
print(f"Saved summary CSV: {summary_csv}")

model, embeddings = create_embeddings(chunks_df)
client, collection = store_in_chroma(chunks_df, embeddings)

_ = test_query(model, collection)

print("\nChunking the scraped documents and creating embeddings are completed")
print("Files created:")
print(f"- {chunks_csv}")
print(f"- {summary_csv}")
print(f"- {CHROMA_DIR}/")
print(f"- Chroma collection: {COLLECTION_NAME}")

Loading source documents...
Loaded 87 source documents
Building chunks...
Created 227 chunks
                      id                                              title  \
0  canada-top-parks_0_c0                                Banff National Park   
1  canada-top-parks_0_c1                                Banff National Park   
2  canada-top-parks_1_c0  What to see while visiting Banff National Park...   
3  canada-top-parks_2_c0  Getting to Banff from Calgary - Banff National...   
4  canada-top-parks_3_c0       The Bow Valley Parkway - Banff National Park   

   chars  
0   2002  
1   1163  
2   2652  
3   1479  
4   1483  
Saved chunks CSV: chunks_embeddings_outputs/parks_chunks.csv
Saved summary CSV: chunks_embeddings_outputs/parks_summary.csv
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Creating/using Chroma DB at: chroma_parks_db
Stored 227 chunks in Chroma collection: parks_rag_v1

Test query: What parks mention camping discounts or park passes?

Result 1
Title: Hiking - Waterton Lakes National Park
Source: Canada
File: canada-top-parks.json
URL: https://parks.canada.ca/pn-np/ab/waterton/activ/experiences/randonee-hiking
Chunk Preview: Hiking - Waterton Lakes National Park

Activities by season Activities in the community Fishing regulations Horseback riding Interpretive programs A park less travelled Wildflower viewing Wildlife watching Winter activities ...

Result 2
Title: Activities and experiences - Pacific Rim National Park Reserve
Source: Canada
File: canada-top-parks.json
URL: https://parks.canada.ca/pn-np/bc/pacificrim/activ
Chunk Preview: Activities and experiences - Pacific Rim National Park Reserve

A custodial group means a group affiliated with an institution, where at least one person is below the age of majority and a minor is unaccompanied by his/he

### Validate the CSV row count, unique chunk ID count, and Chroma collection count

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="chroma_parks_db")
collection = client.get_collection("parks_rag_v1")

print("Count:", collection.count())

sample = collection.get(limit=3)
print(sample["ids"])
print(sample["documents"][0][:300])
print(sample["metadatas"][0])

Count: 227
['canada-top-parks_0_c0', 'canada-top-parks_0_c1', 'canada-top-parks_1_c0']
Banff National Park

Banff National Park Rocky Mountain peaks, glacial lakes, and adventure come together in Banff National Park - Canada's first national park and the flagship of the nation's park system. Banff is part of the Canadian Rocky Mountain Parks UNESCO World Heritage Site. Canada Strong P
{'title': 'Banff National Park', 'url': 'https://parks.canada.ca/pn-np/ab/banff', 'source': 'Canada', 'doc_id': 'canada-top-parks_0', 'file': 'canada-top-parks.json'}


In [ ]:
all_data = collection.get(include=["documents", "metadatas"])
print(len(all_data["ids"]))

227


In [ ]:
import pandas as pd
import chromadb

df = pd.read_csv("chunks_embeddings_outputs/parks_chunks.csv")
print("CSV rows:", len(df))
print("Unique chunk IDs in CSV:", df["id"].nunique())

dupes = df[df["id"].duplicated(keep=False)].sort_values("id")
print(dupes[["id", "title"]].head(20))
print("Duplicate-ID rows:", len(dupes))

client = chromadb.PersistentClient(path="chroma_parks_db")
collection = client.get_collection("parks_rag_v1")
print("Chroma count:", collection.count())

CSV rows: 227
Unique chunk IDs in CSV: 227
Empty DataFrame
Columns: [id, title]
Index: []
Duplicate-ID rows: 0
Chroma count: 227


In [ ]:
# 1: Install required dependencies
!pip install -U langchain-google-genai
!pip install langchain_community
!pip install langchain_core
!pip install pypdf
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires openteleme

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
^C


In [ ]:
# 2: Securely input the Google Gemini API Key
import os
import getpass

# This will prompt you to paste the key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("API Key loaded successfully!")

Enter your Google Gemini API key: ··········
API Key loaded successfully!


Hybrid Park Ranger Chatbot

In [ ]:
# hybrid_chatbot.py
# Hybrid RAG chatbot logic using Gemini + Chroma
# First search QA pairs, then fallback to vector search on full documents.
# Includes chat memory and source tracking.

import sys
import subprocess
import json
import os
import re
import getpass
import time
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


def ensure_google_genai():
    try:
        from google import genai
        return genai
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "google-genai"])
        from google import genai
        return genai


ensure_package("sentence_transformers", "sentence-transformers")
ensure_package("chromadb")
ensure_package("pandas")
ensure_package("numpy")
genai = ensure_google_genai()

from sentence_transformers import SentenceTransformer
import chromadb


if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("API Key loaded successfully!")


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DIR = "chroma_parks_db"
COLLECTION_NAME = "parks_rag_v1"

INPUT_FILES = [
    "canada-top-parks.json",
    "us-top-parks.json",
]


QA_DATA_PATH = "qa-combined-top-parks.csv"
CHAT_MEMORY_PATH = "chunks_embeddings_outputs/chat_memory.json"

TOP_K_QA = 3
TOP_K_DOCS = 4
QA_CONFIDENCE_THRESHOLD = 0.72
DOC_CONFIDENCE_THRESHOLD = 0.55

GEMINI_MODEL = "gemini-2.5-flash"

SYSTEM_PROMPT = """
You are a helpful Park Ranger RAG chatbot for park information in Canada and the US.

Rules:
1. Answer only from the provided retrieved context and recent chat history.
2. If the answer is not clearly supported by the context, say that the information was not found in the retrieved data.
3. Keep answers concise, factual, and user-friendly.
4. When possible, mention whether the answer came from a QA match or document retrieval.
"""


def normalize_text(text: str) -> str:
    text = str(text)
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)
    if a_norm == 0 or b_norm == 0:
        return 0.0
    return float(np.dot(a, b) / (a_norm * b_norm))


def load_json(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def save_json(path: str, data):
    os.makedirs(Path(path).parent, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


class HybridRAGChatbot:
    def __init__(
        self,
        qa_data_path: str = QA_DATA_PATH,
        chroma_dir: str = CHROMA_DIR,
        collection_name: str = COLLECTION_NAME,
        embed_model_name: str = EMBED_MODEL,
        memory_path: str = CHAT_MEMORY_PATH,
    ):
        self.qa_data_path = qa_data_path
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name
        self.memory_path = memory_path

        self.embedder = SentenceTransformer(embed_model_name)
        self.client = chromadb.PersistentClient(path=self.chroma_dir)
        self.collection = self.client.get_collection(name=self.collection_name)

        self.qa_df = self.load_qa_dataset()
        self.qa_embeddings = self.build_qa_embeddings(self.qa_df) if len(self.qa_df) > 0 else None

        self.chat_history = load_json(self.memory_path, [])
        self.llm_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

    def load_qa_dataset(self) -> pd.DataFrame:
        if not os.path.exists(self.qa_data_path):
            print(f"Warning: QA dataset not found at {self.qa_data_path}")
            columns = ["question", "answer", "source_page", "source_url"]
            return pd.DataFrame(columns=columns)

        df = pd.read_csv(self.qa_data_path)

        required_cols = {"question", "answer"}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"QA dataset missing required columns: {missing}")

        if "source_page" not in df.columns:
            df["source_page"] = ""
        if "source_url" not in df.columns:
            df["source_url"] = ""

        df["question"] = df["question"].fillna("").map(normalize_text)
        df["answer"] = df["answer"].fillna("").map(normalize_text)
        df["source_page"] = df["source_page"].fillna("").map(normalize_text)
        df["source_url"] = df["source_url"].fillna("").map(normalize_text)

        df = df[df["question"].str.len() > 0].reset_index(drop=True)
        return df

    def build_qa_embeddings(self, qa_df: pd.DataFrame) -> np.ndarray:
        questions = qa_df["question"].tolist()
        return np.array(self.embedder.encode(questions, show_progress_bar=True))

    def retrieve_from_qa(self, user_query: str, top_k: int = TOP_K_QA) -> Dict[str, Any]:
        if self.qa_embeddings is None or len(self.qa_df) == 0:
            return {"matched": False, "score": 0.0, "results": []}

        query_embedding = np.array(self.embedder.encode([user_query])[0])

        scores = []
        for idx, row_embedding in enumerate(self.qa_embeddings):
            score = cosine_similarity(query_embedding, row_embedding)
            scores.append((idx, score))

        scores.sort(key=lambda x: x[1], reverse=True)
        top_hits = scores[:top_k]

        results = []
        for idx, score in top_hits:
            row = self.qa_df.iloc[idx]
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "source_page": row["source_page"],
                "source_url": row["source_url"],
                "score": round(score, 4),
                "retrieval_type": "qa"
            })

        best_score = results[0]["score"] if results else 0.0
        matched = best_score >= QA_CONFIDENCE_THRESHOLD
        return {"matched": matched, "score": best_score, "results": results}

    def retrieve_from_docs(self, user_query: str, top_k: int = TOP_K_DOCS) -> Dict[str, Any]:
        query_embedding = self.embedder.encode([user_query]).tolist()

        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=top_k,
        )

        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])

        doc_results = []
        for i, (doc, meta) in enumerate(zip(docs, metas)):
            distance = None
            if distances and len(distances[0]) > i:
                distance = distances[0][i]

            similarity_proxy = round(1 - distance, 4) if distance is not None else None

            doc_results.append({
                "text": doc,
                "title": meta.get("title", ""),
                "url": meta.get("url", ""),
                "source": meta.get("source", ""),
                "file": meta.get("file", ""),
                "doc_id": meta.get("doc_id", ""),
                "score": similarity_proxy,
                "retrieval_type": "document"
            })

        best_score = 0.0
        if doc_results and doc_results[0]["score"] is not None:
            best_score = doc_results[0]["score"]

        matched = best_score >= DOC_CONFIDENCE_THRESHOLD
        return {"matched": matched, "score": best_score, "results": doc_results}

    def get_recent_chat_history(self, turns: int = 6) -> List[Dict[str, str]]:
        return self.chat_history[-turns:]

    def build_context(self, qa_result: Dict[str, Any], doc_result: Dict[str, Any]) -> Dict[str, Any]:
        if qa_result["matched"]:
            context_blocks = []
            for item in qa_result["results"]:
                context_blocks.append(
                    f"[QA SOURCE]\n"
                    f"Question: {item['question']}\n"
                    f"Answer: {item['answer']}\n"
                    f"Page: {item['source_page']}\n"
                    f"URL: {item['source_url']}\n"
                    f"Score: {item['score']}"
                )
            return {
                "mode": "qa",
                "context_text": "\n\n".join(context_blocks),
                "sources": qa_result["results"]
            }

        context_blocks = []
        for item in doc_result["results"]:
            context_blocks.append(
                f"[DOCUMENT SOURCE]\n"
                f"Title: {item['title']}\n"
                f"URL: {item['url']}\n"
                f"Source: {item['source']}\n"
                f"Text: {item['text']}\n"
                f"Score: {item['score']}"
            )

        return {
            "mode": "document",
            "context_text": "\n\n".join(context_blocks),
            "sources": doc_result["results"]
        }

    def generate_answer(self, user_query: str, context: str, memory: List[Dict[str, str]], retrieval_mode: str) -> str:
        # 1. Define memory_text
        memory_text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in memory])

        # 2. Check for empty context
        if not context or context.strip() == "":
            return "I haven't found specific information about that in my records. You might want to check the official Parks Canada or National Park Service websites for the most up-to-date information on this topic."

        # 3. Define prompt
        prompt = f"""
{SYSTEM_PROMPT}

Recent chat history:
{memory_text if memory_text else 'No previous conversation.'}

Retrieval mode used: {retrieval_mode}

User question:
{user_query}

Retrieved context:
{context}

Now generate the final answer for the user.
"""

        # 4. Retry loop (This handles the API call and returns the final result)
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = self.llm_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt
                )
                return response.text.strip()
            except Exception as e:
                if "503" in str(e) and attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 2
                    print(f"\nServer busy. Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    return "I'm having trouble connecting to my brain right now. The servers are busy—please try again in a moment!"

        # If the loop finishes without returning, you might want a final fallback
        return "I'm sorry, I couldn't get a response from the server after several attempts."

    def ask(self, user_query: str) -> Dict[str, Any]:
        user_query = normalize_text(user_query)

        qa_result = self.retrieve_from_qa(user_query)
        doc_result = {"matched": False, "score": 0.0, "results": []}

        if not qa_result["matched"]:
            doc_result = self.retrieve_from_docs(user_query)

        chosen_context = self.build_context(qa_result, doc_result)
        recent_memory = self.get_recent_chat_history()

        final_answer = self.generate_answer(
            user_query=user_query,
            context=chosen_context["context_text"],
            memory=recent_memory,
            retrieval_mode=chosen_context["mode"]
        )

        self.chat_history.append({"role": "user", "content": user_query})
        self.chat_history.append({"role": "assistant", "content": final_answer})
        save_json(self.memory_path, self.chat_history)

        return {
            "question": user_query,
            "answer": final_answer,
            "retrieval_mode": chosen_context["mode"],
            "qa_score": qa_result["score"],
            "doc_score": doc_result["score"],
            "sources": chosen_context["sources"]
        }

    def clear_memory(self):
        self.chat_history = []
        save_json(self.memory_path, self.chat_history)


if __name__ == "__main__":
    bot = HybridRAGChatbot()

    print("Park Ranger chatbot ready.")
    print("Type 'exit' to stop, or 'clear' to clear chat memory.\n")

    while True:
        question = input("You: ").strip()

        if question.lower() == "exit":
            break
        if question.lower() == "clear":
            bot.clear_memory()
            print("Chat memory cleared.\n")
            continue

        result = bot.ask(question)

        print("\nPark Ranger:", result["answer"])
        print("\nRetrieval mode:", result["retrieval_mode"])
        print("QA score:", result["qa_score"])
        print("Doc score:", result["doc_score"])
        print("Sources:")
        for i, src in enumerate(result["sources"], start=1):
            if src["retrieval_type"] == "qa":
                print(f"  {i}. [QA] {src['source_page']} | {src['source_url']} | score={src['score']}")
            else:
                print(f"  {i}. [DOC] {src['title']} | {src['url']} | score={src['score']}")
        print()


API Key loaded successfully!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Park Ranger chatbot ready.
Type 'exit' to stop, or 'clear' to clear chat memory.

You: immersive Deep-Sea Reveries multimedia

Park Ranger: The immersive Deep-Sea Reveries multimedia experience at the Marine Environment Discovery Centre has a duration of 20 minutes. (QA match)

Retrieval mode: qa
QA score: 0.7439
Doc score: 0.0
Sources:
  1. [QA] https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/decouverte-tours |  | score=0.7439
  2. [QA] https://parks.canada.ca/amnc-nmca/qc/saguenay/activ/decouverte-tours |  | score=0.3458
  3. [QA] https://www.nps.gov/acad/planyourvisit/west-side-mountains-lakes.htm |  | score=0.2923

You: Bow Valley Parkway

Park Ranger: The Bow Valley Parkway is a 48-kilometer scenic, winding paved route in Banff National Park, Canada, connecting Banff and Lake Louise. It offers a quieter alternative to the Trans-Canada Highway and features roadside pull-offs, picnic areas, and scenic views (Document retrieval).

From 2022 to 2024, Banff National Park piloted a 